In [1]:
import numpy as np
import pandas as pd

from stanbkt.models.standard import StandardBKT

/home/sid/Desktop/Research/StanBKT/src/stanbkt/models/standard.py:235: SyntaxWarning: invalid escape sequence '\m'
  """
/home/sid/Desktop/Research/StanBKT/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


def sim_simple_BKT(
    rng=np.random.default_rng(0),
    nStudents: int = 10,
    nProblems: int = 20,
    nKcs: int = 1,
    prior=0.1,
    learn=0.01,
    forget=0.05,
    guess=0.2,
    slip=0.1,
    kc_sequence=None,
):

    def _param_to_vec(x, name):
        arr = np.asarray(x, dtype=float)
        if arr.size == 1:
            arr = np.repeat(arr, nKcs)
        if arr.size != nKcs:
            raise ValueError(f"{name} must be scalar or length nKcs")
        return arr

    prior_vec = _param_to_vec(prior, "prior")
    learn_vec = _param_to_vec(learn, "learn")
    forget_vec = _param_to_vec(forget, "forget")
    guess_vec = _param_to_vec(guess, "guess")
    slip_vec = _param_to_vec(slip, "slip")

    if kc_sequence is None:
        kc_sequence = rng.integers(0, nKcs, size=nProblems)
    else:
        kc_sequence = np.asarray(kc_sequence, dtype=int)
        if kc_sequence.shape[0] != nProblems:
            raise ValueError("kc_sequence must have length nProblems")
        if kc_sequence.min() < 0 or kc_sequence.max() >= nKcs:
            raise ValueError("kc_sequence entries must be in [0, nKcs-1]")

    knowledge = rng.random(size=(nStudents, nKcs)) < prior_vec
    correctness = np.zeros((nStudents, nProblems), dtype=int)
    states = np.zeros((nStudents, nProblems), dtype=int)

    for t in range(nProblems):
        kc = kc_sequence[t]
        for s in range(nStudents):
            knows_before = knowledge[s, kc]
            if knows_before:
                correct = int(rng.random() >= slip_vec[kc])
            else:
                correct = int(rng.random() < guess_vec[kc])

            correctness[s, t] = correct

            if knows_before:
                knowledge[s, kc] = rng.random() >= forget_vec[kc]
            else:
                knowledge[s, kc] = rng.random() < learn_vec[kc]

            states[s, t] = knowledge[s, kc]

    return (
        correctness,
        states,
        kc_sequence,
        nStudents,
        nProblems,
        nKcs,
    )

In [3]:
# 1) Generate synthetic data (from scratch.ipynb logic)
(
    correctness,
    states,
    kc_sequence,
    n_students,
    n_problems,
    n_kcs,
 ) = sim_simple_BKT(
    rng=np.random.default_rng(123),
    nStudents=50,
    nProblems=30,
    nKcs=3,
    prior=0.4,
    learn=0.04,
    forget=0.01,
    guess=0.2,
    slip=0.1,
    kc_sequence=None,
)

correctness.shape, n_kcs

((50, 30), 3)

In [4]:
# 2) Build long-format dataframe expected by StanBKT
student_idx, problem_idx = np.indices(correctness.shape)

data_df = pd.DataFrame(
    {
        "student_id": student_idx.ravel().astype(str),
        "problem_id": problem_idx.ravel().astype(str),
        "correct": correctness.ravel().astype(int),
        "kc_id": kc_sequence[problem_idx.ravel()].astype(str),
    }
)

data_df.head()

,student_id,problem_id,correct,kc_id
0,0,0,0,0
1,0,1,0,2
2,0,2,1,1
3,0,3,0,0
4,0,4,1,2


In [5]:
data_df['kc_id'].value_counts()

kc_id
0    600
2    550
1    350
Name: count, dtype: int64

In [6]:
# 3) Instantiate and fit the StandardBKT model
model = StandardBKT()
model.fit(data_df)

17:31:43 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/3500 [00:00<?, ?it/s, (Warmup)]





chain 1:   3%|▎         | 100/3500 [00:00<00:10, 314.82it/s, (Warmup)]



chain 1:   6%|▌         | 200/3500 [00:00<00:09, 356.52it/s, (Warmup)]




chain 1:   9%|▊         | 300/3500 [00:00<00:07, 402.29it/s, (Warmup)]


chain 1:  11%|█▏        | 400/3500 [00:00<00:06, 460.94it/s, (Warmup)]


chain 1:  17%|█▋        | 600/3500 [00:01<00:05, 551.97it/s, (Warmup)]


chain 1:  20%|██        | 700/3500 [00:01<00:04, 604.65it/s, (Warmup)]

chain 1:  23%|██▎       | 800/3500 [00:01<00:04, 617.58it/s, (Warmup)]


chain 1:  26%|██▌       | 900/3500 [00:01<00:04, 600.94it/s, (Warmup)]


chain 1:  29%|██▊       | 1000/3500 [00:01<00:04, 610.92it/s, (Warmup)]

chain 1:  31%|███▏      | 1100/3500 [00:02<00:03, 627.25it/s, (Warmup)]


chain 1:  34%|███▍      | 1200/3500 [00:02<00:03, 668.56it/s, (Warmup)]

chain 1:  37%|███▋      | 1300/3500 [00:02<00:03, 692.53it/s, (Warmup)]





17:31:50 - cmdstanpy - INFO - CmdStan done processing.
17:31:50 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/3500 [00:00<?, ?it/s, (Warmup)]


chain 1:   3%|▎         | 100/3500 [00:00<00:08, 387.49it/s, (Warmup)]


chain 1:   6%|▌         | 200/3500 [00:00<00:07, 455.62it/s, (Warmup)]


chain 1:   9%|▊         | 300/3500 [00:00<00:06, 483.57it/s, (Warmup)]



chain 1:  11%|█▏        | 400/3500 [00:00<00:05, 517.90it/s, (Warmup)]


chain 1:  14%|█▍        | 500/3500 [00:00<00:05, 546.11it/s, (Warmup)]




chain 1:  17%|█▋        | 600/3500 [00:01<00:05, 539.67it/s, (Warmup)]


chain 1:  20%|██        | 700/3500 [00:01<00:05, 526.70it/s, (Warmup)]


chain 1:  23%|██▎       | 800/3500 [00:01<00:05, 539.66it/s, (Warmup)]


chain 1:  26%|██▌       | 900/3500 [00:01<00:04, 539.92it/s, (Warmup)]


chain 1:  29%|██▊       | 1000/3500 [00:01<00:04, 594.67it/s, (Warmup)]


chain 1:  31%|███▏      | 1100/3500 [00:02<00:03, 607.01it/s, (Warmup)]


chain 1:  34%|███▍      | 1200/3500 [00:02<00:03, 625.26it/s, (Warmup)]


chain 1:  37%|███▋      | 1300/3500 [00:02<00:03, 655.13


17:31:56 - cmdstanpy - INFO - CmdStan done processing.
17:31:56 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/3500 [00:00<?, ?it/s, (Warmup)]




chain 1:   6%|▌         | 200/3500 [00:00<00:04, 660.78it/s, (Warmup)]


chain 1:   9%|▊         | 300/3500 [00:00<00:04, 707.77it/s, (Warmup)]


chain 1:  11%|█▏        | 400/3500 [00:00<00:04, 711.12it/s, (Warmup)]


chain 1:  17%|█▋        | 600/3500 [00:00<00:03, 748.17it/s, (Warmup)]


chain 1:  20%|██        | 700/3500 [00:00<00:03, 731.52it/s, (Warmup)]


chain 1:  23%|██▎       | 800/3500 [00:01<00:03, 743.82it/s, (Warmup)]


chain 1:  26%|██▌       | 900/3500 [00:01<00:03, 675.35it/s, (Warmup)]


chain 1:  29%|██▊       | 1000/3500 [00:01<00:03, 688.82it/s, (Warmup)]


chain 1:  31%|███▏      | 1100/3500 [00:01<00:03, 684.58it/s, (Warmup)]


chain 1:  34%|███▍      | 1200/3500 [00:01<00:03, 703.75it/s, (Warmup)]



chain 1:  37%|███▋      | 1300/3500 [00:01<00:03, 714.29it/s, (Warmup)]


chain 1:  40%|████      | 1400/3500 [00:01<00:02, 731.24it/s, (Warmup)]


chain 1:  43%|████▎     | 1500/3500 [00:02<00:02, 723.


17:32:02 - cmdstanpy - INFO - CmdStan done processing.


In [7]:
# 4) Predict hidden states
hidden_state_df = model.predict(data_df)
hidden_state_df

AttributeError: 'StandardBKT' object has no attribute '_is_fitted'

In [ ]:
# 5) Predict smoothed hidden states
smoothed_state_df = model.predict_smoothed_states(data_df)
smoothed_state_df

16:11:38 - cmdstanpy - INFO - Chain [1] start processing
16:11:38 - cmdstanpy - INFO - Chain [2] start processing
16:11:38 - cmdstanpy - INFO - Chain [3] start processing
16:11:38 - cmdstanpy - INFO - Chain [4] start processing
16:11:38 - cmdstanpy - INFO - Chain [2] done processing
16:11:38 - cmdstanpy - INFO - Chain [3] done processing
16:11:38 - cmdstanpy - INFO - Chain [1] done processing
16:11:38 - cmdstanpy - INFO - Chain [4] done processing
16:11:38 - cmdstanpy - WARNING - Sample doesn't contain draws from warmup iterations, rerun sampler with "save_warmup=True".
16:11:39 - cmdstanpy - INFO - Chain [1] start processing
16:11:39 - cmdstanpy - INFO - Chain [2] start processing
16:11:39 - cmdstanpy - INFO - Chain [3] start processing
16:11:39 - cmdstanpy - INFO - Chain [4] start processing
16:11:39 - cmdstanpy - INFO - Chain [2] done processing
16:11:39 - cmdstanpy - INFO - Chain [1] done processing
16:11:39 - cmdstanpy - INFO - Chain [4] done processing
16:11:39 - cmdstanpy - INFO

,variable,mean,std,median,2.5%,97.5%,kc_id
0,"know_probs[1,1]",0.009950,0.006642,0.008556,0.001478,0.026687,0
1,"know_probs[1,2]",0.005131,0.003698,0.004250,0.000686,0.014689,0
2,"know_probs[1,3]",0.005087,0.003758,0.004197,0.000651,0.015005,0
3,"know_probs[1,4]",0.000785,0.000651,0.000602,0.000082,0.002523,0
4,"know_probs[1,5]",0.000141,0.000138,0.000098,0.000011,0.000514,0
...,...,...,...,...,...,...,...
1495,"know_probs[50,3]",0.004938,0.008211,0.002453,0.000037,0.024761,1
1496,"know_probs[50,4]",0.001063,0.001662,0.000542,0.000018,0.005498,1
1497,"know_probs[50,5]",0.000493,0.000833,0.000216,0.000008,0.002727,1
1498,"know_probs[50,6]",0.000568,0.000920,0.000263,0.000008,0.002914,1


In [ ]:
correctness[0]

array([0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 1, 1, 0, 1, 0])

In [ ]:
hidden_state_df.loc[hidden_state_df['variable'].str.startswith('pKnow[1,'),'mean'].reset_index(drop=True)

0     0.525289
1     0.145790
2     0.029013
3     0.128348
4     0.025949
5     0.007587
6     0.004560
7     0.004054
8     0.003968
9     0.003952
10    0.003949
11    0.003949
12    0.570393
13    0.210962
14    0.523402
15    0.809726
16    0.938776
17    0.737385
18    0.908555
19    0.969587
20    0.987662
21    0.992735
22    0.956362
23    0.538887
24    0.812045
25    0.926728
26    0.961134
27    0.970531
28    0.973080
29    0.973784
Name: mean, dtype: float64

In [ ]:
smoothed_state_df.loc[smoothed_state_df["variable"].str.startswith("know_probs[1,"), "mean"]

0       0.009950
1       0.005131
2       0.005087
3       0.000785
4       0.000141
5       0.000041
6       0.000024
7       0.000022
8       0.000023
9       0.000033
10      0.000099
11      0.000559
600     0.862694
601     0.957017
602     0.979210
603     0.984352
604     0.984673
605     0.993372
606     0.995463
607     0.995616
608     0.994139
609     0.987611
610     0.987750
1150    0.995981
1151    0.997493
1152    0.997470
1153    0.995984
1154    0.989719
1155    0.963798
1156    0.852494
Name: mean, dtype: float64